# Module 05 — Workflow: routing

**THE ONE IDEA:** classify first, then dispatch to the cheapest handler that can do the
job. **This is the single biggest cost lever in an LLM system**, and it is a workflow —
you wrote the routing table, not the model.

Three tiers here, spanning about a 100x cost range:

| tier | handler | cost |
|---|---|---|
| `lookup` | local Llama on your own M1 | **free** |
| `standard` | `gpt-4.1-mini` | cheap |
| `complex` | `claude-opus-5` | expensive, reasons first |


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client, available
from pydantic import BaseModel
from typing import Literal

print("providers:", available())
mini_c,  MINI,  _ = get_client("openai")
local_c, LOCAL, _ = get_client("local")
opus_c,  OPUS,  _ = get_client("anthropic")

QUESTIONS = [
    "What is the standard variable rate?",                      # lookup
    "Summarise the early repayment charges for a 5-year fix.",  # standard
    "A self-employed applicant with 2 years of accounts wants 92% LTV on a "
    "flat above a shop. Walk through the policy conflicts.",    # complex
]

class Route(BaseModel):
    tier: Literal["lookup", "standard", "complex"]

## The classifier

One cheap call decides the tier. Note it is constrained to three values by `Literal`, so
it cannot invent a fourth route and fall off the table.

In [ ]:
def classify(q):
    s = Route.model_json_schema(); s["additionalProperties"] = False
    r = mini_c.chat.completions.create(
        model=MINI, max_tokens=50,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "route", "strict": True, "schema": s}},
        messages=[{"role": "user", "content":
                   "Classify this banking question.\n"
                   "lookup = one fact. standard = normal explanation. "
                   "complex = multi-policy reasoning.\n\n" + q}],
    )
    return Route.model_validate_json(r.choices[0].message.content).tier

for q in QUESTIONS:
    print(f"{classify(q):9} <- {q[:58]}")

## The dispatch table

Plain Python. Each tier maps to a client, a model, and a price.

In [ ]:
PRICES = {"lookup": (0.0, 0.0), "standard": (0.40, 1.60), "complex": (5.0, 25.0)}
HANDLER = {"lookup": (local_c, LOCAL), "standard": (mini_c, MINI), "complex": (opus_c, OPUS)}

def answer(q, tier):
    client, model = HANDLER[tier]
    if tier == "complex":
        r = client.messages.create(model=model, max_tokens=2000,
                                   messages=[{"role": "user", "content": q}])
        text = "".join(b.text for b in r.content if b.type == "text")
        return text, r.usage.input_tokens, r.usage.output_tokens
    r = client.chat.completions.create(model=model, max_tokens=500,
                                       messages=[{"role": "user", "content": q}])
    return (r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens)

rows = []
for q in QUESTIONS:
    tier = classify(q)
    text, tin, tout = answer(q, tier)
    pin, pout = PRICES[tier]
    rows.append((tier, tin, tout, tin * pin / 1e6 + tout * pout / 1e6))
    print(f"\n[{tier}] {text[:120].strip()}")

## What routing saved

In [ ]:
routed = sum(r[3] for r in rows)
worst  = sum(r[1] * 5.0 / 1e6 + r[2] * 25.0 / 1e6 for r in rows)   # all on Opus

print(f"{'tier':10} {'in':>6} {'out':>6} {'cost $':>10}")
print("-" * 38)
for tier, tin, tout, cost in rows:
    print(f"{tier:10} {tin:6} {tout:6} {cost:10.5f}")
print("-" * 38)
print(f"{'routed':10} {'':6} {'':6} {routed:10.5f}")
print(f"{'all-opus':10} {'':6} {'':6} {worst:10.5f}")
print(f"\nrouting saved {(1 - routed / worst) * 100:.0f}% on these three questions")
print()
print("LESSON — one cheap classifier in front of a dispatch table is the main cost")
print("lever you have. The expensive model only sees the questions that need it, and")
print("the free local model absorbs the lookups entirely.")
print()
print("It is still a WORKFLOW: you wrote the table, the path is auditable, and the")
print("classifier is constrained to three values so it cannot route off the map.")
print("Scale that saving by your daily request volume before reaching for anything")
print("cleverer. Module 30 shows multi-agent moving this number the WRONG way.")

---

**Next:** `06_workflow_parallel_and_evaluator.ipynb`